<a href="https://colab.research.google.com/github/joannapeniket/optiver-volatility/blob/main/optiver_volatility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/joannapeniket/optiver-volatility/blob/main/optiver_volatility.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Volatility Forecasting from Order-Book Data

Predicting each stock's realised volatility over the next 10 minutes from the previous 10 minutes of order-book and trade data, using Optiver's [Realized Volatility Prediction](https://www.kaggle.com/competitions/optiver-realized-volatility-prediction) dataset.

| | |
|---|---|
| Data | 112 stocks, 3,830 ten-minute windows |
| Target | Realised volatility of the next 10 minutes |
| Metric | RMSPE (root mean squared percentage error) |
| Split | 70 / 10 / 20 train / val / test, grouped by time window |
| Models | Naive persistence → log-linear regression → LightGBM |

## Setup

In [ ]:
import time

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupShuffleSplit

pd.set_option("display.float_format", lambda x: f"{x:.4f}")

In [ ]:
DATA_DIR = "optiver_data"   # local copy of the competition files
N_STOCKS = None             # None = all 112 stocks; set e.g. 20 for a quick run
SEED = 42

np.random.seed(SEED)

Data is copied from Google Drive (`My Drive/optiver`) to local disk, which is much faster to read.

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")

SRC = "/content/drive/MyDrive/optiver"
if not os.path.exists(f"{DATA_DIR}/train.csv"):
    os.makedirs(DATA_DIR, exist_ok=True)
    !cp -r "{SRC}/train.csv" "{SRC}/book_train.parquet" "{SRC}/trade_train.parquet" {DATA_DIR}/
print(os.listdir(DATA_DIR))
print(len(os.listdir(f"{DATA_DIR}/book_train.parquet")), "stocks")

In [ ]:
def rmspe(y, p):
    # Root mean squared percentage error: misses on calm stocks count as much as on volatile ones
    y, p = np.asarray(y), np.asarray(p)
    return float(np.sqrt(np.mean(((y - p) / y) ** 2)))

## 1. Data

`train.csv` holds the target for each (stock, window). The book and trade files hold the first 10 minutes of each window; prices are normalised to 1 at the start.

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
stock_ids = sorted(train.stock_id.unique())
if N_STOCKS:
    stock_ids = stock_ids[:N_STOCKS]
    train = train[train.stock_id.isin(stock_ids)].reset_index(drop=True)
print(f"{len(train):,} rows | {train.stock_id.nunique()} stocks | {train.time_id.nunique()} time windows")

book0 = pd.read_parquet(f"{DATA_DIR}/book_train.parquet/stock_id={stock_ids[0]}")
trade0 = pd.read_parquet(f"{DATA_DIR}/trade_train.parquet/stock_id={stock_ids[0]}")
display(book0.head())
display(trade0.head())
train.target.describe()

In [ ]:
def add_wap(book):
    # Weighted average price: leans towards the thinner side of the book,
    # since that side is the one most likely to be taken next
    b = book
    b["wap1"] = (b.bid_price1 * b.ask_size1 + b.ask_price1 * b.bid_size1) / (b.bid_size1 + b.ask_size1)
    b["wap2"] = (b.bid_price2 * b.ask_size2 + b.ask_price2 * b.bid_size2) / (b.bid_size2 + b.ask_size2)
    return b


book0 = add_wap(book0)
tid = book0.time_id.iloc[0]
w = book0[book0.time_id == tid]

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(w.seconds_in_bucket, w.bid_price1, lw=0.8, label="best bid")
ax.plot(w.seconds_in_bucket, w.ask_price1, lw=0.8, label="best ask")
ax.plot(w.seconds_in_bucket, w.wap1, lw=1.2, label="WAP")
ax.set(xlabel="seconds into window", ylabel="normalised price", title=f"Stock {stock_ids[0]}, time_id {tid}")
ax.legend()
plt.show()

## 2. Features

One row per (stock, window), built only from the first 10 minutes. Realised volatility is the root sum of squared log returns of the WAP.

In [ ]:
def book_features(stock_id):
    b = pd.read_parquet(f"{DATA_DIR}/book_train.parquet/stock_id={stock_id}")
    b = add_wap(b)

    # Log returns of the WAP within each window
    b["r1"] = np.log(b.wap1).groupby(b.time_id).diff()
    b["r2"] = np.log(b.wap2).groupby(b.time_id).diff()
    b["r1_sq"], b["r2_sq"] = b.r1 ** 2, b.r2 ** 2

    # Liquidity: relative spread, total size at the top two levels, and how one-sided the book is
    b["spread"] = (b.ask_price1 - b.bid_price1) / ((b.ask_price1 + b.bid_price1) / 2)
    b["depth"] = b.bid_size1 + b.ask_size1 + b.bid_size2 + b.ask_size2
    b["imbalance"] = (b.bid_size1 + b.bid_size2 - b.ask_size1 - b.ask_size2).abs() / b.depth

    out = b.groupby("time_id").agg(
        rv1=("r1_sq", "sum"), rv2=("r2_sq", "sum"),
        spread=("spread", "mean"), depth=("depth", "mean"),
        imbalance=("imbalance", "mean"), n_updates=("r1", "size"),
    )

    # Volatility over the last 300s and last 150s, so recent activity can be weighted more
    for s in (300, 450):
        late = b[b.seconds_in_bucket >= s].groupby("time_id")
        out[f"rv1_last{600 - s}"] = late.r1_sq.sum()
        out[f"rv2_last{600 - s}"] = late.r2_sq.sum()

    rv_cols = [c for c in out if c.startswith("rv")]
    out[rv_cols] = np.sqrt(out[rv_cols])  # sum of squared returns -> realised volatility
    return out.reset_index().assign(stock_id=stock_id)


def trade_features(stock_id):
    t = pd.read_parquet(f"{DATA_DIR}/trade_train.parquet/stock_id={stock_id}")
    t["r_sq"] = np.log(t.price).groupby(t.time_id).diff() ** 2
    out = t.groupby("time_id").agg(
        trade_rv=("r_sq", "sum"), n_trades=("price", "size"),
        trade_size=("size", "sum"), trade_orders=("order_count", "sum"),
    )
    out["trade_rv"] = np.sqrt(out.trade_rv)
    return out.reset_index().assign(stock_id=stock_id)


def features_for_stock(stock_id):
    return book_features(stock_id).merge(trade_features(stock_id), on=["stock_id", "time_id"], how="left")

In [ ]:
t0 = time.time()
feat = pd.concat(Parallel(n_jobs=-1)(delayed(features_for_stock)(s) for s in stock_ids), ignore_index=True)
print(f"Features for {len(stock_ids)} stocks in {time.time() - t0:.0f}s")

df = train.merge(feat, on=["stock_id", "time_id"], how="left")
trade_cols = ["trade_rv", "n_trades", "trade_size", "trade_orders"]
df[trade_cols] = df[trade_cols].fillna(0)          # no trades in the window = zero activity
df = df.fillna(df.median(numeric_only=True))        # rare windows with too few book updates

# Market-wide regime: average across all stocks in the same window.
# Legitimate, since every stock's first 10 minutes is observed before predicting.
for c in ["rv1", "rv1_last300", "rv1_last150", "spread"]:
    df[f"mkt_{c}"] = df.groupby("time_id")[c].transform("mean")

print(df.shape)
df.head()

Volatility clusters: the last 10 minutes' volatility lines up closely with the next 10 minutes'.

In [ ]:
s = df.sample(min(20000, len(df)), random_state=SEED)
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(s.rv1, s.target, s=2, alpha=0.2)
lims = [s[["rv1", "target"]].min().min(), s[["rv1", "target"]].max().max()]
ax.plot(lims, lims, "k--", lw=1)
ax.set(xscale="log", yscale="log", xlabel="RV, first 10 min (feature)", ylabel="RV, next 10 min (target)")
plt.show()
print(f"Correlation of log RV: {np.corrcoef(np.log(df.rv1), np.log(df.target))[0, 1]:.3f}")

## 3. Train / validation / test split

Split by `time_id`, not by row: all stocks in the same window share market conditions, so a row split would leak the test period into training. The test set is used once, at the end.

In [ ]:
def group_split(frame, test_size, seed):
    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    a, b = next(gss.split(frame, groups=frame.time_id))
    return frame.iloc[a], frame.iloc[b]


trainval, test = group_split(df, 0.20, SEED)
tr, va = group_split(trainval, 0.125, SEED)   # 0.125 * 0.8 = 10% of total

# No time window may appear in more than one set
assert not set(tr.time_id) & set(test.time_id)
assert not set(va.time_id) & set(test.time_id)
assert not set(tr.time_id) & set(va.time_id)
print({k: f"{len(v):,} rows / {v.time_id.nunique()} windows" for k, v in [("train", tr), ("val", va), ("test", test)]})

## 4. Models

**Naive persistence** — next 10 minutes' volatility = last 10 minutes'. Every other model has to beat this.

In [ ]:
results = {}
results["Naive: target = RV(first 10 min)"] = rmspe(test.target, test.rv1)

# Best single scale factor under RMSPE (closed form, fitted on train):
# minimise sum(((y - k x) / y)^2)  ->  k = sum(x/y) / sum((x/y)^2)
q = tr.rv1 / tr.target
k = q.sum() / (q ** 2).sum()
results[f"Naive, scaled (x{k:.3f})"] = rmspe(test.target, k * test.rv1)
pd.Series(results, name="test RMSPE")

**Linear regression in logs** — volatility is multiplicative and heavy-tailed, so each coefficient is an elasticity.

In [ ]:
lin_feats = ["rv1", "rv1_last300", "rv1_last150", "mkt_rv1", "spread"]
Xl = lambda d: np.log(d[lin_feats].clip(lower=1e-8))

lin = LinearRegression().fit(Xl(tr), np.log(tr.target))

# exp(log-space prediction) estimates the median, not the RMSPE-optimal value,
# so rescale by one constant fitted on train (same formula as above)
q = np.exp(lin.predict(Xl(tr))) / tr.target
k_lin = q.sum() / (q ** 2).sum()
results["Linear (log-log), 5 features"] = rmspe(test.target, k_lin * np.exp(lin.predict(Xl(test))))
print(f"retransformation scale factor: {k_lin:.3f}")

display(pd.Series(lin.coef_, index=lin_feats, name="coefficient (elasticity)"))
pd.Series(results, name="test RMSPE")

**LightGBM** — all features plus `stock_id` as a categorical. Early stopping uses the validation set only.

In [ ]:
features = [c for c in df.columns if c not in ("target", "time_id")]


def lgb_X(d):
    X = d[features].copy()
    X["stock_id"] = X.stock_id.astype("category")
    return X


def to_lgb(d, ref=None):
    # Weight 1/y^2 turns squared error into squared percentage error,
    # so training optimises RMSPE directly
    return lgb.Dataset(lgb_X(d), d.target, weight=1 / d.target ** 2, reference=ref,
                       categorical_feature=["stock_id"])


params = dict(objective="rmse", learning_rate=0.05, num_leaves=63, min_data_in_leaf=100,
              feature_fraction=0.8, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
              seed=SEED, verbose=-1)

dtr = to_lgb(tr)
dva = to_lgb(va, ref=dtr)
model = lgb.train(params, dtr, num_boost_round=3000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(100), lgb.log_evaluation(200)])

pred_lgb = model.predict(lgb_X(test), num_iteration=model.best_iteration)
results["LightGBM, all features"] = rmspe(test.target, pred_lgb)
pd.Series(results, name="test RMSPE")

## 5. Analysis

In [ ]:
imp = pd.Series(model.feature_importance("gain"), index=features).sort_values()
imp = imp / imp.sum()
imp.tail(15).plot.barh(figsize=(6, 5), title="LightGBM gain importance (share)")
plt.show()

Gain importance splits credit between correlated features, so the ablation retrains without each feature group and measures how much test error rises.

In [ ]:
groups = {
    "market-wide (mkt_*)": [c for c in features if c.startswith("mkt_")],
    "trades": ["trade_rv", "n_trades", "trade_size", "trade_orders"],
    "spread/depth/imbalance": ["spread", "depth", "imbalance"],
    "stock_id": ["stock_id"],
}


def fit_without(drop):
    keep = [c for c in features if c not in drop]

    def ds(d, ref=None):
        X = d[keep].copy()
        cat = ["stock_id"] if "stock_id" in keep else []
        if cat:
            X["stock_id"] = X.stock_id.astype("category")
        return lgb.Dataset(X, d.target, weight=1 / d.target ** 2, reference=ref, categorical_feature=cat), X

    dtr_, _ = ds(tr)
    dva_, _ = ds(va, dtr_)
    _, Xte = ds(test)
    m = lgb.train(params, dtr_, 3000, valid_sets=[dva_], callbacks=[lgb.early_stopping(100, verbose=False)])
    return rmspe(test.target, m.predict(Xte, num_iteration=m.best_iteration))


base = results["LightGBM, all features"]
abl = pd.DataFrame({g: {"test RMSPE without": (r := fit_without(cols)), "change vs full model": r - base}
                    for g, cols in groups.items()}).T
abl.sort_values("change vs full model", ascending=False)

Errors by how volatile each window turned out to be. Positive signed error = over-prediction.

In [ ]:
ev = test[["stock_id", "time_id", "target", "rv1"]].copy()
ev["pred"] = pred_lgb
ev["pct_err"] = (ev.pred - ev.target) / ev.target
ev["target_decile"] = pd.qcut(ev.target, 10, labels=False) + 1

by_dec = ev.groupby("target_decile").agg(
    rmspe=("pct_err", lambda e: np.sqrt(np.mean(e ** 2))),
    mean_signed_pct_err=("pct_err", "mean"))
display(by_dec)

worst = ev.groupby("stock_id").pct_err.apply(lambda e: np.sqrt(np.mean(e ** 2))).sort_values(ascending=False)
print("Hardest stocks (test RMSPE):")
display(worst.head(5))

## 6. Results

In [ ]:
summary = pd.Series(results, name="test RMSPE").to_frame()
naive = results["Naive: target = RV(first 10 min)"]
summary["improvement vs naive"] = (1 - summary["test RMSPE"] / naive).map("{:.1%}".format)
summary